In [1]:
from app.config import CONFIG

#### 1.Open a real browser using Playwright.

In [2]:
import playwright 
from playwright.async_api import async_playwright   

login_url = CONFIG['login_url']

playwright = await async_playwright().start() 
browser = await playwright.chromium.launch(headless=False) 
  


#### 2.Navigate to the login page.

In [3]:
page = await browser.new_page()  
await page.goto(login_url)
print(await page.title())   

Login :: Damn Vulnerable Web Application (DVWA)


#### 3.Fill all fields provided in login_steps.


In [4]:
login_steps = CONFIG['login_steps']
# Query fields from the web page
fields = await page.query_selector_all("input")

for field in fields:
    name = await field.get_attribute("name")
    id = await field.get_attribute("id")
    type = await field.get_attribute("type")
    print(f"Field name: {name}, id: {id}, type: {type}")

# Input user data    
for input in login_steps:
    print(input)
    await page.fill(input['selector'], input['value'])

Field name: username, id: None, type: text
Field name: password, id: None, type: password
Field name: Login, id: None, type: submit
Field name: user_token, id: None, type: hidden
{'selector': "[name='username']", 'value': 'admin'}
{'selector': "[name='password']", 'value': 'password'}


#### 4.Click the submit_selector

In [5]:
await page.click(CONFIG["submit_selector"])
await page.wait_for_load_state("networkidle")                                                                  
print("URL after login:", page.url)

URL after login: http://localhost:8080/index.php


#### 5.Validate that login was successful.Use

In [6]:
# Check if the url is the same as the start_url_after_login
if page.url == CONFIG['start_url_after_login']:
    print("Login successful")
else:
    print("Login failed")


Login successful


#### 6. Save the authenticated browser storage / session state.

In [21]:
storage_state = await browser.contexts[0].storage_state()
state_logged_in = {
    "url": page.url,
    "cookies": storage_state['cookies'],
    "local_storage": storage_state['origins'],
}



#### 7.Start crawling from start_url_after_login

#### 8. Extract links, forms, buttons, scripts, and relevant page

In [43]:
# Worker functions
page_url = page.url

links = await page.query_selector_all("a[href]")
links = [await link.get_attribute("href") for link in links]

forms = await page.query_selector_all("form")
forms = [await form.get_attribute("action") for form in forms]

buttons = await page.query_selector_all("button, input[type='submit'], input[type='button']")
buttons = [await button.inner_text() for button in buttons]

scripts = await page.query_selector_all("script[src]")
scripts = [await script.get_attribute("src") for script in scripts]



#### 9.Capture all relevant browser requests and responses.


In [ ]:
context = browser.contexts[0]                                                                                  
requests = []                
context.on("request", lambda req: requests.append({                                                            
      "method": req.method,
      "url": req.url,                                                                                            
      "resource_type": req.resource_type,
  }))

context.on("response", lambda res: requests.append({                                                           
      "url": res.url,
      "status": res.status,                                                                                      
  }))             

#### 10.Store all discovered pages, forms, links, and network traffic in

#### 11.Avoid duplicate links.Deduplicate

#### 12.Avoid duplicate forms.Deduplicate

##### 13. Avoid duplicate captured requests.Use a hash over method + URL + relevant headers

#### 14.Respect allowed_domains.

#### 15.Respect exclude_patterns.Skip

#### 16.Support crawling depth limits.

#### 17.Support maximum page limits.

#### 18.Support asynchronous crawling using asyncio.

#### 19.Allow the script to be stopped at any moment.

#### 20.Resume from the last saved state when re-run with the same